In [5]:
%pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")
print(dataset)

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  259kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [6]:
train = dataset["train"]
first_record = train[0]

tokens = first_record["tokens"]
ner_tag_ids = first_record["ner_tags"]

print(tokens)
print(ner_tag_ids)

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
[3, 0, 7, 0, 0, 0, 7, 0, 0]


In [7]:
tag_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC",
]

gold_tags = [tag_names[tag_id] for tag_id in ner_tag_ids]
print(gold_tags)

['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


In [1]:
def bio_to_spans(tags):
    spans = []
    i = 0

    while i < len(tags):
        tag = tags[i]
        if tag == "O":
            i += 1
            continue

        start = i
        entity_type = tag[2:]
        i += 1

        while i < len(tags) and tags[i] == f"I-{entity_type}":
            i += 1

        spans.append((start, i, entity_type))

    return spans

In [8]:
gold_spans = bio_to_spans(gold_tags)
print(gold_spans)

for start, end, entity_type in gold_spans:
    text = " ".join(tokens[start:end])
    print(text, entity_type)

[(0, 1, 'ORG'), (2, 3, 'MISC'), (6, 7, 'MISC')]
EU ORG
German MISC
British MISC


In [4]:
def count_matches(gold_tags, predicted_tags):
    if len(gold_tags) != len(predicted_tags):
        raise ValueError("Число меток в двух разметках должно совпадать.")

    gold = set(bio_to_spans(gold_tags))
    predicted = set(bio_to_spans(predicted_tags))

    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    return tp, fp, fn


def scores_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1

In [9]:
fake_tags = ["B-ORG", "O", "B-LOC", "O", "O", "O", "O", "O", "O"]

tp, fp, fn = count_matches(gold_tags, fake_tags)
print(tp, fp, fn)

1 1 2


In [10]:
precision, recall, f1 = scores_from_counts(tp, fp, fn)

print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1: {f1:.2%}")

Precision: 50.00%
Recall: 33.33%
F1: 40.00%


In [12]:
validation = dataset["validation"]
total_tp, total_fp, total_fn = 0, 0, 0

for row_index in range(3):
    record = validation[row_index]
    row_gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]
    row_fake_tags = ["O"] * len(record["tokens"])

    tp, fp, fn = count_matches(row_gold_tags, row_fake_tags)
    total_tp += tp
    total_fp += fp
    total_fn += fn

print(total_tp, total_fp, total_fn)

0 0 6
